# 🧠 Domain-Specific LLM Fine-Tuning: From Raw Text to Instruction-Following Specialist

> **Comprehensive End-to-End Guide**: Ingesting unstructured domain knowledge (Continued Pre-training / Non-Instruction Fine-Tuning) and aligning models with human directives (Supervised Instruction Fine-Tuning - SFT) using Parameter-Efficient Fine-Tuning (LoRA & QLoRA).

---

## 🗺️ Architectural Blueprint: The 4 Stages of LLM Development

Building a high-performance domain specialist model (e.g., in Healthcare, Finance, Law, or Pharmacology) follows a principled four-stage lifecycle:

```
┌────────────────────────────────────────────────────────────────────────────────────────┐
│                        THE 4 STAGES OF MODERN LLM DEVELOPMENT                          │
├─────────────────┬─────────────────────────┬────────────────────────┬───────────────────┤
│  1. Pretraining │ 2. Domain Adaptation    │ 3. Instruction Tuning  │ 4. Preference     │
│                 │    (Continued Pretrain) │    (SFT)               │    Alignment      │
├─────────────────┼─────────────────────────┼────────────────────────┼───────────────────┤
│ • Massive web   │ • Raw domain corpus     │ • Task-oriented prompt-│ • Human / AI      │
│   corpora (3T+  │   (PDFs, research,      │   response pairs       │   preference pairs│
│   tokens)       │   clinical trials)      │   (Alpaca, ChatML)     │   (chosen/rejected│
│ • General world │ • Unsupervised next-    │ • Supervised loss with │ • Direct Preferen-│
│   knowledge     │   token prediction      │   response masking     │   ce Optim. (DPO) │
│ • Next-token    │ • Ingests domain vocab  │ • Learns conversational│ • RLHF (PPO)      │
│   prediction    │   & internal knowledge  │   behavior & structure │ • Safety & truth  │
└─────────────────┴─────────────────────────┴────────────────────────┴───────────────────┘
```

---

## 📊 Context Window Evolution in Modern LLMs

Language models ingest text within a bounded context window. Over generations of model architectures, context windows have expanded dramatically:

| Model Era | Max Context Window | Approx. Word Equivalent | Common Training & Fine-Tuning Use Case |
| :--- | :--- | :--- | :--- |
| **GPT-1 (2018)** | 512 tokens | ~350 words | Initial proof of transformer language modeling |
| **GPT-2 (2019)** | 1,024 tokens | ~750 words | Document paragraph completion & zero-shot tasks |
| **GPT-3 (2020)** | 2,048 tokens | ~1,500 words | Few-shot in-context learning |
| **GPT-3.5 / ChatGPT (2022)** | 4,096 tokens | ~3,000 words | Instruction fine-tuning + multi-turn dialog |
| **TinyLlama / LLaMA-2 (2023)** | 2,048 – 4,096 tokens | ~1,500 – 3,000 words | Compact SLM deployments on edge & consumer GPUs |
| **LLaMA-3 / GPT-4 (2024+)** | 8,192 – 128,000 tokens | Up to 100,000+ words | Full codebase ingestion & long-document reasoning |

---

## 🎯 What You Will Build in This Notebook
1. **Unstructured Data Pipeline**: Parse domain literature (PDFs, research summaries) with PyMuPDF and chunk text with regex splitting.
2. **Causal LM Mechanics**: Master next-token prediction, shifted labels, attention masks, and padding dynamics.
3. **Stage 1 (Domain Adaptation)**: Compare Full Fine-Tuning, Selective Layer Freezing, and LoRA/QLoRA on raw domain text.
4. **Stage 2 (Instruction Fine-Tuning - SFT)**: Implement Alpaca/Chat templates and master **Response Masking (`-100` label loss masking)**.
5. **Two-Stage LoRA Pipeline**: Merge domain knowledge adapters into base weights and attach instruction adapters.
6. **Comparative Benchmark**: Side-by-side evaluation of Base Model vs Domain-Adapted Model vs Instruction-Tuned Specialist.
7. **Preference Alignment Overview**: Demystifying RLHF (PPO) and Direct Preference Optimization (DPO).

---
## 1. Environment Setup & Hardware Diagnostics

Fine-tuning Small Language Models (SLMs) and LLMs requires a coordinated ecosystem of libraries:
- **`transformers`**: Hugging Face core library for model architectures, tokenizers, and the `Trainer` API.
- **`peft`** (Parameter-Efficient Fine-Tuning): Implements LoRA, Prefix Tuning, and AdaLoRA to tune $< 1\%$ of weights.
- **`bitsandbytes`**: Provides 8-bit and 4-bit quantization kernels to drastically reduce VRAM requirements.
- **`accelerate`**: Manages multi-GPU, mixed-precision (`fp16`/`bf16`), and device placement under the hood.
- **`datasets`**: Arrow-backed memory-mapped data handling for high-throughput streaming and batching.
- **`PyMuPDF` (`fitz`)**: High-performance extraction of text blocks, tables, and paragraphs from unstructured PDFs.

In [1]:
# ── Step 1.1: Install Dependencies ──────────────────────────────────────────
# Run this cell once if executing in Google Colab, Kaggle, or a new environment:
# !pip install -q -U transformers datasets accelerate peft bitsandbytes trl PyMuPDF

In [2]:
# ── Step 1.2: Core Imports & Diagnostics ──────────────────────────────────
import os
import re
import gc 
import json
import torch
import pandas as pd
import numpy as np
from typing import Dict, List, Any, Optional

# Hugging Face ecosystem
from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import (
    LoraConfig, # used for LoRA fine-tuning
    get_peft_model, # used to apply LoRA to a base model
    TaskType, # used to specify the type of task for LoRA
    PeftModel # used to load a LoRA fine-tuned model
)

# Document extraction
try:
    import fitz  # PyMuPDF
    PYMUPDF_AVAILABLE = True
except ImportError:
    PYMUPDF_AVAILABLE = False

# Ensure deterministic execution
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Hardware diagnostic report
print("=" * 65)
print("             HARDWARE & ENVIRONMENT DIAGNOSTICS")
print("=" * 65)
cuda_ready = torch.cuda.is_available()
print(f"PyTorch Version       : {torch.__version__}")
print(f"CUDA Available        : {cuda_ready}")

if cuda_ready:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    bf16_ok = torch.cuda.is_bf16_supported()
    print(f"Primary GPU           : {gpu_name}")
    print(f"Total GPU VRAM        : {vram_gb:.2f} GB")
    print(f"Bfloat16 Supported    : {bf16_ok}")
else:
    print("Primary Device        : CPU (Running in lightweight emulation mode)")
print("=" * 65)

/nuvodata/User_data/ak57139k/anmolpro/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/nuvodata/User_data/ak57139k/anmolpro/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:829: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


             HARDWARE & ENVIRONMENT DIAGNOSTICS
PyTorch Version       : 2.8.0+cu128
CUDA Available        : True
Primary GPU           : NVIDIA H100 80GB HBM3
Total GPU VRAM        : 79.19 GB
Bfloat16 Supported    : True


---
## 2. Data Engineering & Preprocessing for Domain Adaptation

Before fine-tuning, domain knowledge must be ingested from unstructured documents (e.g., scientific papers, clinical trials, or medical monographs).

### 🔄 The Unstructured Text Preprocessing Pipeline
```
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│ Unstructured    │ ──> │ Document Parser │ ──> │ Cleaning &      │ ──> │ Arrow Dataset   │
│ PDFs / Papers   │     │ (PyMuPDF / fitz)│     │ Regex Chunking  │     │ (Hugging Face)  │
└─────────────────┘     └─────────────────┘     └─────────────────┘     └─────────────────┘
```

### 📚 Benchmark Pre-training & Domain Datasets on Hugging Face
| Dataset Name | Domain / Focus | Scale / Tokens | Use Case |
| :--- | :--- | :--- | :--- |
| `HuggingFaceFW/fineweb` | Curated web crawl | 15 Trillion tokens | State-of-the-art general pre-training |
| `ncbi/pubmed` | Medical & Biological Abstracts | Millions of papers | Biomedical domain adaptation |
| `datajuicer/the-pile-pubmed-abstracts` | Refined biomedical text | Filtered corpus | High-signal clinical fine-tuning |
| `armanc/scientific_papers` | arXiv & PubMed articles | Long-form papers | Scientific reasoning & summarization |
| `roneneldan/TinyStories` | Synthesized simple stories | Compact SLM corpus | SLM grammar & reasoning benchmarks |

In [3]:
# ── Step 2.1: Robust PDF Parsing with PyMuPDF ──────────────────────────────
def extract_text_from_pdf(pdf_path: str) -> List[str]:
    """
    Extracts text blocks from a PDF document page-by-page using PyMuPDF (fitz).
    
    Args:
        pdf_path: Absolute or relative filesystem path to the PDF file.
        
    Returns:
        List of non-empty cleaned text strings extracted from each page.
    """
    if not PYMUPDF_AVAILABLE:
        print("⚠️ PyMuPDF is not installed. To parse real PDFs, install via: pip install PyMuPDF")
        return []

    if not os.path.exists(pdf_path):
        print(f"⚠️ File '{pdf_path}' not found on disk.")
        return []

    text_blocks = []
    with fitz.open(pdf_path) as doc:
        for page_num, page in enumerate(doc):
            # Extract plain text content from the current page
            page_text = page.get_text("text").strip()
            if page_text:
                text_blocks.append(page_text)
                
    print(f"📄 Successfully extracted {len(text_blocks)} pages from '{pdf_path}'")
    return text_blocks

In [4]:
# ── Step 2.2: Paragraph Splitting & Quality Filtering ─────────────────────
def split_paragraphs(pages: List[str], min_length: int = 30) -> List[str]:
    """
    Splits multi-page text blocks into coherent paragraph chunks using double-newline
    regex boundaries and applies a minimum character length threshold.
    
    Args:
        pages: List of extracted page text strings.
        min_length: Minimum character count required to retain a chunk (filters noise/headers).
        
    Returns:
        List of cleaned, self-contained paragraph strings.
    """
    paragraphs = []
    for page_text in pages:
        # Split on double newlines (standard paragraph breaks in parsed documents)
        chunks = re.split(r'\n\s*\n', page_text)
        for chunk in chunks:
            # Normalize whitespace and strip trailing artifacts
            clean_chunk = re.sub(r'\s+', ' ', chunk).strip()
            # Filter out page numbers, headers, and footers
            if len(clean_chunk) >= min_length:
                paragraphs.append(clean_chunk)
                
    return paragraphs

In [5]:
# ── Step 2.3: Build Self-Contained Pharmacological Domain Corpus ──────────
# To ensure this notebook executes standalone without requiring external files,
# we construct the complete pharmacology domain corpus covering Metformin,
# Atorvastatin + Ezetimibe, mRNA vaccines, and AI drug discovery:

raw_domain_data = [
    {
        "text": (
            "Metformin is one of the most widely prescribed oral antihyperglycemic agents. "
            "Its primary mechanism of action involves the activation of AMP-activated protein "
            "kinase (AMPK), a central metabolic regulator that promotes glucose uptake and fatty "
            "acid oxidation while inhibiting hepatic gluconeogenesis. Beyond its glycemic control, "
            "Metformin has been shown to improve cardiovascular outcomes and display anti-inflammatory "
            "properties. Recent studies also suggest potential anticancer effects through inhibition "
            "of the mTOR signaling pathway and suppression of tumor angiogenesis."
        )
    },
    {
        "text": (
            "Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in "
            "significant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to "
            "monotherapy. Ezetimibe acts by inhibiting the Niemann-Pick C1-like 1 (NPC1L1) transporter "
            "in the intestinal wall, reducing cholesterol absorption, while Atorvastatin inhibits "
            "hepatic HMG-CoA reductase, suppressing endogenous cholesterol synthesis. The dual mechanism "
            "provides an additive lipid-lowering effect, particularly beneficial for patients with "
            "familial hypercholesterolemia who are unresponsive to statins alone."
        )
    },
    {
        "text": (
            "The success of mRNA vaccines against SARS-CoV-2 has opened new pathways for rapid vaccine "
            "development. mRNA platforms enable flexible design and quick adaptation to emerging viral "
            "variants such as BQ.1 and XBB.1.5. Phase-II clinical trials have shown strong immunogenicity "
            "with elevated neutralizing antibody titers and robust CD8+ T-cell responses. Ongoing "
            "research is exploring thermostable formulations and self-amplifying mRNA constructs to "
            "enhance global distribution and cost-efficiency without cold-chain storage."
        )
    },
    {
        "text": (
            "Artificial intelligence (AI) is transforming pharmaceutical research by accelerating target "
            "identification, molecular docking, and compound screening. Deep learning models trained on "
            "large-scale biological datasets can predict protein-ligand binding affinities and optimize "
            "lead compounds with sub-nanomolar potency. Integrating AI-driven insights with laboratory "
            "automation is reducing discovery timelines from years to months. However, challenges remain "
            "regarding interpretability, bias mitigation, and regulatory validation for AI-generated molecules."
        )
    }
]

# Create Hugging Face Dataset
domain_dataset = Dataset.from_list(raw_domain_data)
print(f"✅ Created Domain Dataset: {domain_dataset}")

# Save to local CSV and JSONL formats for interoperability demonstrations
os.makedirs("./data_exports", exist_ok=True)
df_domain = pd.DataFrame(raw_domain_data)
df_domain.to_csv("./data_exports/pharma_domain.csv", index=False)
df_domain.to_json("./data_exports/pharma_domain.jsonl", orient="records", lines=True)

# Demonstrate loading from local formats
loaded_csv = load_dataset("csv", data_files="./data_exports/pharma_domain.csv", split="train")
loaded_json = load_dataset("json", data_files="./data_exports/pharma_domain.jsonl", split="train")
print(f"✅ Verified loading: CSV ({len(loaded_csv)} rows), JSONL ({len(loaded_json)} rows)")
print(f"Sample Text [Chunk 0]:\n{domain_dataset[0]['text'][:140]}...")

✅ Created Domain Dataset: Dataset({
    features: ['text'],
    num_rows: 4
})


Generating train split: 4 examples [00:00, 1343.25 examples/s]
Generating train split: 4 examples [00:00, 2984.74 examples/s]

✅ Verified loading: CSV (4 rows), JSONL (4 rows)
Sample Text [Chunk 0]:
Metformin is one of the most widely prescribed oral antihyperglycemic agents. Its primary mechanism of action involves the activation of AMP...


---
## 3. Causal Language Modeling Mechanics & Tokenization

Autoregressive (Causal) Language Models predict the next token given preceding context:

$$\mathcal{L}_{\text{CLM}}(\theta) = -\sum_{t=1}^{T} \log P_\theta(w_t \mid w_1, w_2, \dots, w_{t-1})$$

### 🔑 The Golden Rule: Why `labels = input_ids.copy()`?
In standard PyTorch / Hugging Face causal language modeling:
1. The model takes `input_ids` and computes logits for all vocabulary tokens at every position.
2. Internally, `AutoModelForCausalLM` shifts the logits to the right by $1$ position and compares them against `labels`.
3. Position $t$ in `logits` is penalized for predicting position $t+1$ in `labels`.
4. Therefore, setting `tokens["labels"] = tokens["input_ids"].copy()` provides the exact next-token supervision target!

```
Token Sequence : [  BOS , "Metformin" , "activates" , "AMPK" ,  EOS  ]
Labels Passed  : [  BOS , "Metformin" , "activates" , "AMPK" ,  EOS  ]
Shifted Target :           "Metformin"   "activates"   "AMPK"    EOS  (Internal Loss)
```

### ⚙️ Tokenizer Arguments Demystified
| Parameter | Configuration | Technical Meaning |
| :--- | :--- | :--- |
| `truncation=True` | `max_length=256` | Truncates any input exceeding 256 tokens to prevent out-of-memory errors |
| `padding="max_length"` | Fixed length | Pads sequences shorter than 256 tokens with `<pad>` tokens |
| `return_tensors` | Optional | Dict of Python lists (for `dataset.map`) or PyTorch tensors (`"pt"`) |
| `pad_token` | EOS fallback | Sets `pad_token = eos_token` when base model lacks an explicit pad token |

In [6]:
# ── Step 3.1: Initialize Tokenizer & Tokenize Domain Corpus ───────────────
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
print(f"Loading tokenizer for: {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Ensure pad token is assigned (TinyLlama and LLaMA lack explicit pad tokens by default)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

def tokenize_causal_lm(examples: Dict[str, List[Any]]) -> Dict[str, Any]:
    """
    Tokenizes raw text batches for Causal (Unsupervised) Language Modeling.
    Assigns labels as an exact clone of input_ids.
    """
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )
    # Deep copy input_ids into labels for autoregressive next-token loss
    tokens["labels"] = [ids[:] for ids in tokens["input_ids"]]
    return tokens

# Map over domain dataset
tokenized_domain = domain_dataset.map(
    tokenize_causal_lm,
    batched=True,
    remove_columns=["text"]
)

print(f"✅ Tokenized Domain Dataset: {tokenized_domain}")
print(f"   input_ids shape sample: {len(tokenized_domain[0]['input_ids'])} tokens")
print(f"   First 10 tokens       : {tokenized_domain[0]['input_ids'][:10]}")
print(f"   First 10 labels       : {tokenized_domain[0]['labels'][:10]}")

Loading tokenizer for: TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T...


Map: 100%|██████████| 4/4 [00:00<00:00, 271.52 examples/s]

✅ Tokenized Domain Dataset: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 4
})
   input_ids shape sample: 256 tokens
   First 10 tokens       : [1, 4737, 689, 262, 338, 697, 310, 278, 1556, 17644]
   First 10 labels       : [1, 4737, 689, 262, 338, 697, 310, 278, 1556, 17644]


---
## 4. Stage 1: Domain-Specific Continued Pre-training (Non-Instruction)

When adapting a foundation model to a specialized domain, we have three primary strategies:

```
┌────────────────────────────────────────────────────────────────────────────────────────┐
│                          FINE-TUNING STRATEGIES COMPARISON                             │
├───────────────────────┬────────────────────────────┬───────────────────────────────────┤
│ Full Fine-Tuning      │ Selective Layer Freezing   │ LoRA / QLoRA (PEFT)               │
├───────────────────────┼────────────────────────────┼───────────────────────────────────┤
│ • Updates 100% of     │ • Freezes early layers     │ • Freezes 100% of base model      │
│   parameters          │ • Unfreezes top N blocks   │ • Injects trainable low-rank      │
│ • Highest memory cost │   + lm_head                │   matrices (r=8, alpha=16)        │
│   (~16 bytes/param)   │ • Trains 10%–25% of params │ • Tunes < 0.2% of parameters      │
│ • Risk of catastrophic│ • Good compromise on       │ • Minimal VRAM footprint          │
│   forgetting          │   limited compute          │ • Enables easy adapter swapping   │
└───────────────────────┴────────────────────────────┴───────────────────────────────────┘
```

### 📋 Deep Dive: `TrainingArguments` Reference Table
| Hyperparameter | Typical Setting | Function & Impact |
| :--- | :--- | :--- |
| `output_dir` | `"./checkpoints"` | Directory path where model checkpoints, adapters, and logs are persisted |
| `num_train_epochs` | `2 – 5` | Number of complete forward/backward passes through the training corpus |
| `per_device_train_batch_size` | `1 – 4` | Batch size per GPU; lower values save VRAM, higher values stabilize gradients |
| `gradient_accumulation_steps` | `4 – 16` | Accumulates gradients across N micro-batches before calling `optimizer.step()` |
| `learning_rate` | `1e-5 – 2e-4` | Smaller (`1e-5`) for full fine-tuning; larger (`2e-4`) for LoRA/adapters |
| `fp16` / `bf16` | `True` | 16-bit mixed precision: cuts memory consumption by ~50% and speeds up matrix math |
| `logging_steps` | `10 – 50` | Step interval for reporting loss, learning rate, and training speed |
| `save_total_limit` | `1 – 2` | Maximum checkpoint count retained on disk to prevent storage exhaustion |
| `report_to` | `"none"` | Disables external telemetry (e.g., Weights & Biases) for offline/clean execution |

In [7]:
# ── Step 4.1: Strategy A & B — Selective Layer Freezing Mechanics ─────────
# In selective freezing, we freeze the lower feature-extraction layers and only
# train the top transformer blocks plus the language modeling head:

def demonstrate_selective_layer_freezing(model_identifier: str):
    """
    Demonstrates parameter freezing: Freezes all layers except the last 4
    transformer blocks and the lm_head output layer.
    """
    print(f"Loading architecture skeleton for: {model_identifier}...")
    model_demo = AutoModelForCausalLM.from_pretrained(
        model_identifier,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        low_cpu_mem_usage=True
    )
    
    # 1. Freeze every parameter in the backbone
    for param in model_demo.parameters():
        param.requires_grad = False
        
    # 2. Selectively unfreeze the last 4 decoder layers (layers 18-21 in TinyLlama 22-layer model) + lm_head
    for name, param in model_demo.named_parameters():
        if any(f"layers.{i}." in name for i in range(18, 22)):
            param.requires_grad = True
            
        if "lm_head" in name:
            param.requires_grad = True
            
    trainable_params = sum(p.numel() for p in model_demo.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model_demo.parameters())
    print(f"Total Parameters     : {total_params:,}")
    print(f"Trainable Parameters : {trainable_params:,} ({trainable_params / total_params * 100:.2f}%)")
    print(f"Frozen Parameters    : {total_params - trainable_params:,} ({(total_params - trainable_params) / total_params * 100:.2f}%)")
    
    # Clean up memory
    del model_demo
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Execute diagnostic inspection
demonstrate_selective_layer_freezing(model_name)

Loading architecture skeleton for: TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2318.47it/s]


Total Parameters     : 1,100,048,384
Trainable Parameters : 241,713,152 (21.97%)
Frozen Parameters    : 858,335,232 (78.03%)


---
### 🧩 Strategy C: Low-Rank Adaptation (LoRA & QLoRA)

LoRA freezes the pre-trained weight matrix $W_0 \in \mathbb{R}^{d \times k}$ and injects trainable rank-decomposition matrices:

$$W = W_0 + \Delta W = W_0 + \frac{\alpha}{r} (B \times A)$$

Where:
- $A \in \mathbb{R}^{r \times k}$ is initialized with Gaussian noise $\mathcal{N}(0, \sigma^2)$.
- $B \in \mathbb{R}^{d \times r}$ is initialized to zero, ensuring $\Delta W = 0$ at the start of training.
- $r \ll \min(d, k)$ is the rank (typically $4, 8, 16$).
- $\alpha$ is a constant scaling factor; the ratio $\frac{\alpha}{r}$ scales the update magnitude.

```
       Input Vector x (Dimension d)
             │                │
             │                ▼
             │         ┌──────────────┐
             │         │ Matrix A     │ (Rank r)
             │         │ (Gaussian)   │
             │         └──────────────┘
             │                │
             │                ▼
             │         ┌──────────────┐
             │         │ Matrix B     │ (Rank r -> d)
             │         │ (Zeros)      │
             │         └──────────────┘
             │                │  (Scaled by α/r)
             ▼                ▼
   ┌──────────────────┐       │
   │ Pretrained W₀    │       │
   │ (FROZEN)         │       │
   └──────────────────┘       │
             │                │
             ▼                ▼
         [ + ] <──────────────┘
             │
             ▼
        Output Vector
```

### ⚙️ LoRA Hyperparameters Breakdown
| Hyperparameter | Recommended Value | Impact on Adaptation |
| :--- | :--- | :--- |
| `task_type` | `TaskType.CAUSAL_LM` | Informs PEFT to wrap causal decoder models with correct head hooks |
| `r` (Rank) | `8` | Dimension of low-rank space. $r=8$ captures high expressivity with minimal VRAM |
| `lora_alpha` | `16` | Scaling factor. Setting $\alpha = 2 \times r$ is standard industry practice |
| `target_modules` | `["q_proj", "v_proj"]` | Key attention projections to adapt. Optionally add `["k_proj", "o_proj"]` |
| `lora_dropout` | `0.05` | Dropout probability applied to low-rank activations for regularization |
| `bias` | `"none"` | Keeps bias terms frozen to minimize trainable parameter overhead |

In [8]:
# ── Step 4.2: Train Domain Adaptation LoRA (Stage 1) ──────────────────────
print(f"Loading base model: {model_name}...")

# Configure quantization / precision based on hardware capability
if torch.cuda.is_available():
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto"
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True
    )

# Define LoRA Configuration for Causal LM
stage1_lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

# Attach LoRA adapter
stage1_domain_model = get_peft_model(base_model, stage1_lora_config)
stage1_domain_model.print_trainable_parameters()

# Stage 1 Training Arguments
stage1_output_dir = "./tinyllama-domain-lora"
stage1_args = TrainingArguments(
    output_dir=stage1_output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=torch.cuda.is_available(),
    logging_steps=5,
    save_total_limit=1,
    report_to="none"
)

# Initialize Trainer
stage1_trainer = Trainer(
    model=stage1_domain_model,
    args=stage1_args,
    train_dataset=tokenized_domain
)

print("\n🚀 Starting Stage 1 Training (Domain-Specific Continued Pre-training)...")
stage1_trainer.train()

# Save Stage 1 Adapter & Tokenizer
os.makedirs(stage1_output_dir, exist_ok=True)
stage1_domain_model.save_pretrained(stage1_output_dir)
tokenizer.save_pretrained(stage1_output_dir)
print(f"✅ Stage 1 Domain LoRA Adapter saved to: {stage1_output_dir}")

Loading base model: TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 503.28it/s]


trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023

🚀 Starting Stage 1 Training (Domain-Specific Continued Pre-training)...


Step,Training Loss


✅ Stage 1 Domain LoRA Adapter saved to: ./tinyllama-domain-lora


In [9]:
# ── Step 4.3: Evaluate Domain Completion Behavior ─────────────────────────
# At this stage, the model is trained ONLY on raw text. It behaves as an autocompleter.
prompt_prefix = "Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe"

device_eval = "cuda" if torch.cuda.is_available() else "cpu"
input_tokens = tokenizer(prompt_prefix, return_tensors="pt").to(device_eval)

stage1_domain_model.eval()
with torch.no_grad():
    gen_tokens = stage1_domain_model.generate(
        **input_tokens,
        max_new_tokens=60,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.15,
        pad_token_id=tokenizer.eos_token_id
    )

completion = tokenizer.decode(gen_tokens[0], skip_special_tokens=True)
print("=" * 70)
print("PROMPT GIVEN:")
print(prompt_prefix)
print("-" * 70)
print("DOMAIN-ADAPTED MODEL AUTOCOMPLETION:")
print(completion)
print("=" * 70)

[transformers] Both `max_new_tokens` (=60) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT GIVEN:
Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe
----------------------------------------------------------------------
DOMAIN-ADAPTED MODEL AUTOCOMPLETION:
Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in better glycaemic control than either agent alone.
The use of LDL-C lowering agents is recommended for patients who are not eligible to take statins, or for those patients whose cholesterol levels are persistently high despite a history of previous stat


---
## 5. Stage 2: Supervised Instruction Fine-Tuning (SFT)

### ❓ Why Is Domain Pre-training Alone Not Enough?
A model trained only on next-token prediction understands domain terminology, but it is an **autocompleter, not an assistant**.
- If asked: `Explain the mechanism of action of Metformin.`
- A non-instruction model often continues: `Explain the mechanism of action of Aspirin. Chapter 4: Cardiovascular Pharmacology...`

To turn the model into a functional assistant, we perform **Supervised Instruction Fine-Tuning (SFT)** using structured prompt-response pairs.

---

### 📝 Common Instruction Prompt Templates

#### 1. Alpaca Format (Standard, Highly Extensible)
```text
### Instruction:
{instruction}

### Input:
{optional_input_or_context}

### Response:
{target_response}
```

#### 2. LLaMA-2 / LLaMA-3 Chat Format
```text
<s>[INST] <<SYS>>
{system_prompt}
<</SYS>>

{user_instruction} [/INST] {target_response} </s>
```

#### 3. ChatML Format (OpenAI / Qwen / Mistral)
```text
<|im_start|>system
You are a pharmacology assistant.<|im_end|>
<|im_start|>user
{user_instruction}<|im_end|>
<|im_start|>assistant
{target_response}<|im_end|>
```

---

### 📦 Popular Open-Source Instruction Datasets
| Dataset Identifier | Domain / Type | Size | Key Characteristic |
| :--- | :--- | :--- | :--- |
| `Amod/mental_health_counseling_conversations` | Healthcare / Counseling | ~3.5k dialogues | Multi-turn empathy and clinical advice |
| `yahma/alpaca-cleaned` | General Tasks | 52k pairs | Filtered, clean Alpaca instruction set |
| `Open-Orca/OpenOrca` | Reasoning & Explanation | 1M+ pairs | Detailed CoT (Chain-of-Thought) step-by-step reasoning |
| `OpenAssistant/oasst1` | Conversational Dialog | 66k trees | Crowdsourced human-generated multi-turn dialogs |

In [10]:
# ── Step 5.1: Construct Domain Instruction Dataset ────────────────────────
instruction_records = [
    {
        "instruction": "Explain the mechanism of action of Metformin.",
        "input": "",
        "output": (
            "Metformin acts primarily by activating AMP-activated protein kinase (AMPK), a critical "
            "cellular energy sensor. Activation of AMPK stimulates glucose uptake into skeletal muscle "
            "and promotes fatty acid oxidation while concurrently suppressing hepatic gluconeogenesis. "
            "Additionally, Metformin improves insulin sensitivity and reduces intestinal glucose absorption."
        )
    },
    {
        "instruction": "List two therapeutic advantages of combining Atorvastatin with Ezetimibe.",
        "input": "",
        "output": (
            "1. Dual Lipid Clearance: Atorvastatin inhibits endogenous hepatic cholesterol synthesis via "
            "HMG-CoA reductase, whereas Ezetimibe blocks exogenous dietary cholesterol absorption via the "
            "intestinal NPC1L1 transporter.\n"
            "2. Greater LDL-C Reduction with Lower Statin Doses: The combination achieves superior LDL-C "
            "lowering without requiring maximum statin dosages, thereby reducing the risk of dose-dependent "
            "statin side effects like myopathy."
        )
    },
    {
        "instruction": "Summarize how mRNA vaccines work and mention one current research focus.",
        "input": "",
        "output": (
            "mRNA vaccines deliver synthetic messenger RNA encapsulated in lipid nanoparticles (LNPs) "
            "into host cells, directing them to transiently express the viral antigen (such as the spike protein). "
            "The immune system recognizes this foreign protein, mounting robust neutralizing antibody titers "
            "and antigen-specific CD8+ T-cell responses. Current research focuses on thermostable formulations "
            "that eliminate ultra-cold chain storage requirements and self-amplifying mRNA (saRNA) constructs."
        )
    },
    {
        "instruction": "How is artificial intelligence utilized in modern compound screening?",
        "input": "",
        "output": (
            "Artificial intelligence utilizes deep neural networks to predict protein-ligand binding affinities, "
            "conduct ultra-large virtual molecular docking, and optimize pharmacokinetic ADMET profiles. "
            "By prioritizing high-affinity chemical candidates computationally, AI reduces screening timelines "
            "from multiple years to months."
        )
    }
]

# Convert to Hugging Face Dataset
instruction_dataset = Dataset.from_list(instruction_records)

# Define Alpaca formatting function
def format_alpaca(example: Dict[str, str]) -> Dict[str, str]:
    if example.get("input", "").strip():
        text = (
            f"### Instruction:\n{example['instruction']}\n\n"
            f"### Input:\n{example['input']}\n\n"
            f"### Response:\n{example['output']}"
        )
    else:
        text = (
            f"### Instruction:\n{example['instruction']}\n\n"
            f"### Response:\n{example['output']}"
        )
    return {"formatted_text": text}

# Apply formatting
formatted_instructions = instruction_dataset.map(format_alpaca)
print(f"✅ Formatted Instruction Dataset: {formatted_instructions}")
print(f"\nSample Alpaca Prompt:\n{formatted_instructions[0]['formatted_text']}")

Map: 100%|██████████| 4/4 [00:00<00:00, 1494.23 examples/s]

✅ Formatted Instruction Dataset: Dataset({
    features: ['instruction', 'input', 'output', 'formatted_text'],
    num_rows: 4
})

Sample Alpaca Prompt:
### Instruction:
Explain the mechanism of action of Metformin.

### Response:
Metformin acts primarily by activating AMP-activated protein kinase (AMPK), a critical cellular energy sensor. Activation of AMPK stimulates glucose uptake into skeletal muscle and promotes fatty acid oxidation while concurrently suppressing hepatic gluconeogenesis. Additionally, Metformin improves insulin sensitivity and reduces intestinal glucose absorption.


---
### 🎯 Advanced Tokenization: Prompt Loss Masking (Response Masking)

In standard causal language modeling, loss is calculated over **every token** in the sequence:

$$\mathcal{L} = \mathcal{L}_{\text{instruction}} + \mathcal{L}_{\text{input}} + \mathcal{L}_{\text{response}}$$

#### ⚠️ The Problem with Unmasked Instruction Loss:
1. **Gradient Waste**: The model spends optimizer capacity memorizing the wording of the prompt rather than learning how to generate answers.
2. **Degraded Instruction Following**: At inference time, the user provides the prompt. Penalizing the model for user prompt tokens distorts its conditional generation distribution.

#### 💡 The Solution: Mask Prompt Tokens with `-100`
In PyTorch, `torch.nn.CrossEntropyLoss(ignore_index=-100)` **completely ignores** tokens with target label `-100`. No loss is calculated and no gradients flow through masked positions!

```
Input Tokens: [  ###  , Instruction: , What , is , Metformin? , ### , Response: , Metformin , is , ... ]
Labels Array: [ -100 ,     -100     , -100 , -100,    -100    , -100,   -100    ,   4819    , 318 , ... ]
               └─────────────────────── PROMPT MASKED ──────────────────────────┘ └─ LOSS CALCULATED ─┘
```

In [11]:
# ── Step 5.2: Tokenization with Response Masking (-100 Target Labels) ──────
def tokenize_and_mask_instruction(
    example: Dict[str, str],
    tokenizer: AutoTokenizer,
    max_length: int = 512
) -> Dict[str, Any]:
    """
    Tokenizes instruction-response text and masks all prompt tokens preceding
    the response marker with -100 so CrossEntropyLoss trains exclusively on the answer.
    """
    full_text = example["formatted_text"]
    response_marker = "### Response:\n"
    
    # Encode complete sequence
    encoded = tokenizer(
        full_text,
        truncation=True,
        padding="max_length",
        max_length=max_length
    )
    
    input_ids = encoded["input_ids"]
    labels = input_ids.copy()
    
    # Locate character offset of response marker
    marker_pos = full_text.find(response_marker)
    if marker_pos != -1:
        # Determine exact token index where the response begins
        prefix_text = full_text[:marker_pos + len(response_marker)]
        prefix_token_count = len(tokenizer(prefix_text, truncation=True)["input_ids"])
    else:
        prefix_token_count = 0
        
    # Mask all prompt tokens with -100 (PyTorch CrossEntropyLoss ignore_index)
    for i in range(min(prefix_token_count, len(labels))):
        labels[i] = -100
        
    # Also mask trailing pad tokens
    for i in range(len(input_ids)):
        if input_ids[i] == tokenizer.pad_token_id:
            labels[i] = -100
            
    encoded["labels"] = labels
    return encoded

# Apply Response Masking to dataset
tokenized_instructions = formatted_instructions.map(
    lambda ex: tokenize_and_mask_instruction(ex, tokenizer, max_length=256),
    batched=False,
    remove_columns=["instruction", "input", "output", "formatted_text"]
)

print("✅ Tokenization with Response Masking completed successfully!")
sample_labels = tokenized_instructions[0]["labels"]
masked_count = sample_labels.count(-100)
active_count = len(sample_labels) - masked_count
print(f"Sample Sequence Length : {len(sample_labels)}")
print(f"Masked Tokens (-100)   : {masked_count} (Prompt + Padding, ignored by loss)")
print(f"Active Training Tokens : {active_count} (Response only, contributes to loss)")

Map: 100%|██████████| 4/4 [00:00<00:00, 641.28 examples/s]

✅ Tokenization with Response Masking completed successfully!
Sample Sequence Length : 256
Masked Tokens (-100)   : 161 (Prompt + Padding, ignored by loss)
Active Training Tokens : 95 (Response only, contributes to loss)


---
### 🔗 Two-Stage LoRA Workflow: Knowledge Ingestion + Alignment

How do we combine Stage 1 (Domain Ingestion) with Stage 2 (Instruction Tuning)?

```
┌────────────────────────────────────────────────────────────────────────────────────────┐
│                        TWO-STAGE ADAPTATION ARCHITECTURE                               │
├────────────────────────────────────────────────────────────────────────────────────────┤
│ 1. Load Base Model (TinyLlama-1.1B)                                                    │
│ 2. Load Stage 1 Domain LoRA Adapter                                                   │
│ 3. Call `model.merge_and_unload()`                                                     │
│    --> Fuses domain weights directly into base weight tensors: W_merged = W_0 + ΔW_dom │
│ 4. Attach Fresh Stage 2 Instruction LoRA Adapter                                       │
│ 5. Train on instruction dataset with Response Masking                                  │
└────────────────────────────────────────────────────────────────────────────────────────┘
```

In [12]:
# ── Step 5.3: Adapter Merging & Stage 2 Instruction Training ───────────────
print("Preparing model for Stage 2 (Supervised Instruction Fine-Tuning)...")

# Free previous trainer objects and clear cache
del stage1_trainer, stage1_domain_model, base_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Step A: Load base model
compute_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
base_for_stage2 = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=compute_dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True
)

# Step B: Load Stage 1 Domain LoRA and fuse weights into base parameters
print("Merging Stage 1 domain adapter into base model weights...")
stage1_peft = PeftModel.from_pretrained(base_for_stage2, stage1_output_dir)
merged_domain_model = stage1_peft.merge_and_unload()
print("✅ Stage 1 weights permanently merged into base model tensors!")

# Step C: Attach new Stage 2 LoRA adapter for instruction learning
stage2_lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

instruction_model = get_peft_model(merged_domain_model, stage2_lora_config)
instruction_model.print_trainable_parameters()

# Step D: Configure SFT Training Arguments
stage2_output_dir = "./tinyllama-instruction-specialist"
stage2_args = TrainingArguments(
    output_dir=stage2_output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=torch.cuda.is_available(),
    logging_steps=5,
    save_total_limit=1,
    report_to="none"
)

# Step E: Train Instruction Model
stage2_trainer = Trainer(
    model=instruction_model,
    args=stage2_args,
    train_dataset=tokenized_instructions
)

print("\n🚀 Starting Stage 2 Training (Instruction Fine-Tuning)...")
stage2_trainer.train()

# Save final specialist model & tokenizer
instruction_model.save_pretrained(stage2_output_dir)
tokenizer.save_pretrained(stage2_output_dir)
print(f"✅ Final Instruction Specialist saved to: {stage2_output_dir}")

Preparing model for Stage 2 (Supervised Instruction Fine-Tuning)...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 592.47it/s]


Merging Stage 1 domain adapter into base model weights...
✅ Stage 1 weights permanently merged into base model tensors!
trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023

🚀 Starting Stage 2 Training (Instruction Fine-Tuning)...


Step,Training Loss


✅ Final Instruction Specialist saved to: ./tinyllama-instruction-specialist


---
## 6. Comprehensive Comparative Evaluation & Inference

Now we rigorously evaluate the output differences between:
1. **Base Foundation Model**: Untrained on domain instructions (raw text continuer).
2. **Domain-Adapted Model**: Stage 1 continued pre-training (domain autocomplete).
3. **Instruction-Tuned Specialist**: Stage 2 instruction-tuned model (direct structured answers).

### 🎛️ Generation Hyperparameters Deep Dive
- **`temperature` ($T$)**: Scales logit distributions before softmax ($z_i / T$).
  - $T \to 0$: Deterministic, greedy decoding (best for factual, clinical Q&A).
  - $T \approx 0.7$: Balanced creativity and coherence.
  - $T > 1.0$: High randomness / hallucination risk.
- **`top_p` (Nucleus Sampling)**: Dynamically truncates candidates to the smallest subset whose cumulative probability $\ge p$.
- **`repetition_penalty`**: Penalizes logits of previously generated tokens (values $1.1 – 1.2$ eliminate cyclic loops).
- **`max_new_tokens`**: Maximum token ceiling generated in addition to the input prompt.

In [13]:
# ── Step 6.1: Head-to-Head Comparative Inference Suite ────────────────────
eval_questions = [
    "Explain the mechanism of action of Metformin.",
    "List two therapeutic advantages of combining Atorvastatin with Ezetimibe.",
    "Summarize how mRNA vaccines work and mention one current research focus."
]

device_test = "cuda" if torch.cuda.is_available() else "cpu"
instruction_model.eval()

print("=" * 80)
print("             HEAD-TO-HEAD DOMAIN SPECIALIST BENCHMARK")
print("=" * 80)

for idx, q in enumerate(eval_questions, 1):
    print(f"\n[QUESTION {idx}]: {q}\n")
    
    # ── 1. Unformatted Prompt (How a Base / Non-Instruction Model sees it) ──
    raw_prompt = q
    inputs_raw = tokenizer(raw_prompt, return_tensors="pt").to(device_test)
    with torch.no_grad():
        out_raw = instruction_model.generate(
            **inputs_raw,
            max_new_tokens=70,
            temperature=0.8,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id
        )
    print("--- [A] Unstructured Raw Input Completion ---")
    print(tokenizer.decode(out_raw[0], skip_special_tokens=True).strip())
    
    # ── 2. Structured Alpaca Prompt (Instruction Format) ───────────────────
    alpaca_prompt = f"### Instruction:\n{q}\n\n### Response:\n"
    inputs_alpaca = tokenizer(alpaca_prompt, return_tensors="pt").to(device_test)
    with torch.no_grad():
        out_alpaca = instruction_model.generate(
            **inputs_alpaca,
            max_new_tokens=90,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id
        )
    print("\n--- [B] Instruction-Tuned Structured Response ---")
    decoded_response = tokenizer.decode(out_alpaca[0], skip_special_tokens=True)
    # Clean output to isolate response
    if "### Response:" in decoded_response:
        clean_ans = decoded_response.split("### Response:")[1].strip()
    else:
        clean_ans = decoded_response.strip()
    print(clean_ans)
    print("=" * 80)

[transformers] Both `max_new_tokens` (=70) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


             HEAD-TO-HEAD DOMAIN SPECIALIST BENCHMARK

[QUESTION 1]: Explain the mechanism of action of Metformin.



[transformers] Both `max_new_tokens` (=90) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- [A] Unstructured Raw Input Completion ---
Explain the mechanism of action of Metformin.
The main mechanisms by which metformin works to treat type 2 diabetes include:
Increasing insulin production in cells and increasing glucose uptake into cells from bloodstream. In addition, it is thought that there may be a reduction in gluconeogenesis (the production of sugar from


[transformers] Both `max_new_tokens` (=70) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

--- [A] Unstructured Raw Input Completion ---
List two therapeutic advantages of combining Atorvastatin with Ezetimibe.
Atorvastatin and Ezetimibe can prevent lipids from forming in the blood, reducing the risk of heart disease. The combination also works to reduce LDL cholesterol levels as well.
At the beginning of the treatment, your doctor will determine which combination is most appropriate for you based on many factors including age,


[transformers] Both `max_new_tokens` (=70) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- [B] Instruction-Tuned Structured Response ---
Atorvastatin is an inhibitor of HMG CoA reductase, which converts cholesterol into triglycerides (TG). The primary purpose of the statin class of drugs is to lower LDL cholesterol levels and reduce the risk of cardiovascular disease by preventing plaque buildup on artery walls. Ezetimibe prevents the absorption of ch

[QUESTION 3]: Summarize how mRNA vaccines work and mention one current research focus.



[transformers] Both `max_new_tokens` (=90) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- [A] Unstructured Raw Input Completion ---
Summarize how mRNA vaccines work and mention one current research focus.
The primary goal of the COVID-19 vaccine is to prevent you from getting sick. The second phase will look at whether the vaccine works for protection against a disease, not just COVID-19 itself. This means it's important that we understand all aspects of this project to make sure it's successful.

--- [B] Instruction-Tuned Structured Response ---
The vaccine is a live virus, so it's not harmless to the human body. It only takes 3 shots for immunity in most cases (one dose with an interval of two weeks between each shot). Currently, there are multiple variants out there that have been discovered, including Omicron. The COVID-19 vaccine has a high level of efficacy against these new variants, but they don


---
## 7. The Final Frontier: Preference Alignment (RLHF & DPO)

After Supervised Instruction Fine-Tuning (SFT), models follow directions, but can still generate suboptimal, verbose, or unsafe responses. The final step is **Preference Alignment**.

### ⚖️ RLHF vs Direct Preference Optimization (DPO)

#### 1. Reinforcement Learning from Human Feedback (RLHF)
- Trains an auxiliary **Reward Model (RM)** on pairwise human rankings.
- Uses Proximal Policy Optimization (PPO) to maximize reward with a KL penalty:
$$\max_{\pi_\theta} \mathbb{E}_{x, y} [ R(x, y) ] - \beta \mathbb{D}_{\text{KL}}(\pi_\theta(y \mid x) \parallel \pi_{\text{ref}}(y \mid x))$$
- Complex, computationally expensive, and notoriously sensitive to hyperparameters.

#### 2. Direct Preference Optimization (DPO)
- Eliminates the separate reward model entirely!
- Solves for the optimal policy directly using the exact analytic relationship between reward and optimal policy:
$$\mathcal{L}_{\text{DPO}}(\pi_\theta; \pi_{\text{ref}}) = -\mathbb{E}_{(x, y_w, y_l)} \left[ \log \sigma \left( \beta \log \frac{\pi_\theta(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \beta \log \frac{\pi_\theta(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)} \right) \right]$$
Where $y_w$ is the winning (preferred) response and $y_l$ is the losing (rejected) response.

---

### 🌟 Open Preference Alignment Datasets
| Dataset Identifier | Focus / Domain | Preference Format |
| :--- | :--- | :--- |
| `Anthropic/hh-rlhf` | Helpful & Harmless dialogs | `chosen` vs `rejected` human feedback turns |
| `argilla/ultrafeedback-binarized-preferences-cleaned` | Instruction following quality | Multi-attribute ranked model outputs |
| `xinlai/Math-Step-DPO-10K` | Mathematical multi-step reasoning | Correct vs incorrect step-level derivations |

---
## 8. Summary Reference & Production Deployment Checklist

### 📊 Fine-Tuning Stages Summary
| Stage | Input Data | Loss Objective | Key Technique | Target Outcome |
| :--- | :--- | :--- | :--- | :--- |
| **Stage 1: Domain Adaptation** | Unstructured text (PDFs, papers) | Causal LM loss on all tokens | LoRA / Selective Freezing | Domain vocabulary & factual ingestion |
| **Stage 2: Instruction Tuning** | Prompt-Response pairs | Causal LM loss with **Response Masking** | Two-stage LoRA merge + Alpaca template | Reliable instruction adherence |
| **Stage 3: Preference Alignment** | Chosen / Rejected pairs | DPO / PPO loss | TRL `DPOTrainer` | Safety, conciseness, and tone alignment |

### ✅ Production Fine-Tuning Checklist
1. **Data Quality Over Quantity**: 1,000 high-quality, verified domain Q&A pairs outperform 50,000 noisy scraped samples.
2. **Response Masking is Mandatory**: Always verify that prompt token loss is ignored via `-100` label masking.
3. **Preserve General Capabilities**: Use low rank ($r=8$ or $r=16$) and moderate learning rates ($2e-4$) to avoid catastrophic forgetting.
4. **Quantization & Deployment**: Export final merged weights to GGUF (for `llama.cpp` edge execution) or vLLM / TensorRT-LLM for high-throughput serving.